In [ ]:
import csv
import time
import datetime
from sklearn import metrics

from nltk import tag
from nltk.tag import map_tag
from nltk import word_tokenize
from nltk.tag.mapping import map_tag

from nltk.tag.perceptron import PerceptronTagger

from nltk.corpus import brown as corpus



In [ ]:
corpus_length = len(corpus.fileids())
print(corpus_length)

In [ ]:
# TRAINING

corpus_length = len(corpus.fileids())
training_length = int(corpus_length * 0.8)
# training_length = 10

training_sentences = corpus.tagged_sents(corpus.fileids()[0:training_length])

tagger = PerceptronTagger(load=False)
   
start = time.time()
tagger.train(training_sentences)
end = time.time()

processing_time = end - start

In [ ]:
# TESTING 1:
# Text del corpus de test amb etiquetes originals en una llista de tuples (token, etiqueta)

text_id = corpus.fileids()[training_length:]
text_id = corpus.fileids()[training_length]

testing_sentences = corpus.tagged_sents(text_id)

print("Test set Gold labels for text " + corpus.fileids()[training_length] + ":")
print(testing_sentences[0][0:3],testing_sentences[1][0:3])

# Text del corpus en una llista de paraules

bare_sentences = [([word for word,_ in sentence]) for sentence in testing_sentences]
   # ALTERNATIVA: bare_sentences = corpus.sents(text_id)
    
print(" ")
print("Words:")
print(bare_sentences[0][0:3], bare_sentences[1][0:3])
    

In [ ]:
# TESTING 2: 
# Predicció d'etiquetes del corpus de test

predicted_sentences = [tagger.tag([word for word,_ in sentence]) for sentence in testing_sentences]

print(" ")
print("Predicted labels (tagger default's tagset):")
print(predicted_sentences[0][0:3], predicted_sentences[1][0:3])

# Canvi de tagset: Universal

for i in range(len(predicted_sentences)):
    predicted_sentences[i] = [(word, tag, map_tag('en-ptb', 'universal', tag)) for word, tag in predicted_sentences[i]]

print(" ")
print("Predicted labels (Tagset 'Universal'):")
print(predicted_sentences[0][0:3], predicted_sentences[1][0:3])


In [ ]:
# EVALUATION 1
# Gold and predicted labels of all sentences joined in one single list, respectivament.

golds = [tag for sentence in testing_sentences for _,tag in sentence]
predicted_labels = [tag for sentence in predicted_sentences for _,tag,_ in sentence]

print("Gold labels:")
print(golds[0:3])
print(" ")
print("Predicted labels:")
print(predicted_labels[0:3])



In [ ]:
# EVALUATION 2
# Metrics

for preds,tagger in [(predicted_labels,tagger)]:

    accuracy = metrics.accuracy_score(golds,preds)
    precision = metrics.precision_score(golds,preds,average='weighted')
    recall = metrics.recall_score(golds,preds,average='weighted')
    f1_score = metrics.f1_score(golds,preds,average='weighted')
        
    print("Metrics for",tagger)   
    print(" Accuracy :", accuracy)
    print(" Precision:", precision)
    print(" Recall   :", recall)
    print(" F1-Score :", f1_score)


In [ ]:
# EXPORTACIÓ DE les primeres 15 línies del corpus de test.

now = datetime.datetime.now().strftime("%y%m%d-%H%M")
tagger_name = 'Perceptron'
corpus_name = 'Brown'

export_name = text_id if type(text_id) == str else (str(len(text_id)) + ' texts from ' + (text_id)[0] + ' to ' + (text_id)[-1])

all_labels = testing_sentences[0:15].copy()

with open(now + '-' + tagger_name + '-' + corpus_name + '-' + export_name + '-.csv', mode='w') as dades:
    dades_writer = csv.writer(dades, delimiter=',')
    for i in range(len(testing_sentences[0:15])):
        for j in range (len(testing_sentences[0:15][i])):
            all_labels[i][j] = testing_sentences[0:15][i][j] + predicted_sentences[0:15][i][j]
            dades_writer.writerow(all_labels[i][j])

print(" ")
print("Gold vs predicted labels (Tagset 'Universal'):")
print(all_labels[0][0:3], all_labels[1][0:3])

